<a href="https://colab.research.google.com/github/taselshambakey/DECI-final-project/blob/main/Tasneem_DECI_final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initialization

In [2]:
import csv
import json
import sqlite3

import pandas as pd
from bs4 import BeautifulSoup

DB_PATH = "database.db"
BOOKS_JSON_PATH = "books.json"
KICKOFF_HTML_PATH = "Reading Kickoff signups.html"

TASK1_ANSWERS_PATH = "answers.txt"
TASK1_JSON_PATH = "task1_combined_data.json"
TASK1_CSV_PATH = "task1_combined_data.csv"
TASK2_CSV_PATH = "task2_cleaned_data.csv"

output_lines = []


def log(line=""):
    print(line)
    output_lines.append(str(line))


## Task 1- Goal 1

Five questions answered directly against the database. No cleaning is
performed here -- every query runs against the raw `checkouts` table as-is,
duplicates included. Data cleaning (removing true duplicates, handling
inconsistent spellings, etc.) is deliberately deferred to Task 2.

In [3]:
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

# ---------------------------------------------------------------------------
log("################ Q1: Checkouts per member (incl. zero) ################")
log("Reasoning:")
log("  - Every member must appear in the result, even those with no checkouts,")
log("    so members is the driving table with a LEFT JOIN out to checkouts")
log("    (an INNER JOIN would silently drop members with zero checkouts).")
log("  - COUNT(c.checkout_id) counts only matched checkout rows per member; for")
log("    members with no matches the LEFT JOIN produces NULLs, which COUNT(...)")
log("    correctly reports as 0 rather than NULL.")
log("  - The raw checkouts table is queried directly, with no de-duplication --")
log("    that cleanup is intentionally left for Task 2, so these counts include")
log("    a small number of duplicate-row inflation that Task 2 later corrects.")
log("  - Sorted busiest-first (checkout_count DESC), with member_id as a")
log("    tiebreaker for a stable, reproducible order.")
log()
q1 = """
SELECT m.member_id,
       m.first_name || ' ' || m.last_name AS member_name,
       COUNT(c.checkout_id) AS checkout_count
FROM members m
LEFT JOIN checkouts c ON c.member_id = m.member_id
GROUP BY m.member_id, member_name
ORDER BY checkout_count DESC, m.member_id ASC;
"""
cur.execute(q1)
rows1 = cur.fetchall()
for r in rows1:
    log(dict(r))
log(f"Total members: {len(rows1)}")
log(f"Members with 0 checkouts: {sum(1 for r in rows1 if r['checkout_count'] == 0)}")

# ---------------------------------------------------------------------------
log()
log("################ Q2: Author pattern search ################")
log("Reasoning:")
log("  - Chosen pattern: author's first name starts with 'A' (SQL LIKE 'A%').")
log("    This is an arbitrary but concrete pattern choice, picked because it")
log("    returns a small, easy-to-verify set (3 distinct authors) rather than")
log("    an empty or overwhelming result.")
log("  - Query hits the books table only, since author is a book attribute,")
log("    not a checkout attribute -- no join to checkouts is needed, so")
log("    duplicate checkout rows don't factor into this question at all.")
log("  - Sorted by author then title so same-author books are grouped together.")
log()
pattern = "A%"
q2 = """
SELECT book_id, title, author
FROM books
WHERE author LIKE ?
ORDER BY author, title;
"""
cur.execute(q2, (pattern,))
rows2 = cur.fetchall()
for r in rows2:
    log(dict(r))

# ---------------------------------------------------------------------------
log()
log("################ Q3: Top 5 most-borrowed titles ################")
log("Reasoning:")
log("  - 'Most popular' = highest raw checkout frequency per title, so we")
log("    JOIN checkouts to books on book_id and COUNT checkouts per book.")
log("    An INNER JOIN is correct here (not LEFT JOIN) because a book with")
log("    zero checkouts isn't a 'popular book' and shouldn't appear at all.")
log("  - The raw checkouts table is used as-is, so any duplicate checkout row")
log("    is counted here too -- again, left untouched on purpose for Task 2.")
log("  - ORDER BY times_borrowed DESC, then title ASC as a tiebreaker, then")
log("    LIMIT 5 gives exactly the top five, with a stable, reproducible order")
log("    when counts tie.")
log()
q3 = """
SELECT b.book_id, b.title, b.author, COUNT(c.checkout_id) AS times_borrowed
FROM checkouts c
JOIN books b ON b.book_id = c.book_id
GROUP BY b.book_id, b.title, b.author
ORDER BY times_borrowed DESC, b.title ASC
LIMIT 5;
"""
cur.execute(q3)
rows3 = cur.fetchall()
for r in rows3:
    log(dict(r))

# ---------------------------------------------------------------------------
log()
log("################ Q4: Top 10 most active readers ################")
log("Reasoning:")
log("  - Same shape as Q1 (per-member checkout count), but here we only want")
log("    members who have actually borrowed something, so an INNER JOIN")
log("    (members to checkouts) is used instead of a LEFT JOIN -- a member")
log("    with 0 checkouts can't be one of the 'most active readers'.")
log("  - The raw checkouts table is used, so this ranking may be a little")
log("    inflated by a handful of duplicate rows -- expected at this stage.")
log("  - ORDER BY checkout_count DESC with member_id as a tiebreaker, then")
log("    LIMIT 10, gives the requested top-10 ranking highest to lowest.")
log()
q4 = """
SELECT m.member_id,
       m.first_name || ' ' || m.last_name AS member_name,
       COUNT(c.checkout_id) AS checkout_count
FROM members m
JOIN checkouts c ON c.member_id = m.member_id
GROUP BY m.member_id, member_name
ORDER BY checkout_count DESC, m.member_id ASC
LIMIT 10;
"""
cur.execute(q4)
rows4 = cur.fetchall()
for r in rows4:
    log(dict(r))

# ---------------------------------------------------------------------------
log()
log("################ Q5: Neighborhood activity, skipping 10 most recent ################")
log("Reasoning:")
log("  - Chosen neighborhood: Maadi -- the largest neighborhood by member count,")
log("    which gives a big enough checkout history to demonstrate 'looking")
log("    further back in time'.")
log("  - The raw neighborhood column has inconsistent casing/whitespace")
log("    ('Maadi', 'Maadi ' with a trailing space, etc.), so the WHERE clause")
log("    compares LOWER(TRIM(m.neighborhood)) against a lowercase literal.")
log("    This only affects how rows are MATCHED for this query -- it does not")
log("    change or clean any stored data, so it isn't a data-cleaning step.")
log("  - 'Newest to oldest' means ORDER BY checkout_date DESC (checkout_id DESC")
log("    as a tiebreaker for same-day checkouts, for a stable order).")
log("  - 'Looking past the ten most recent' means skipping the first 10 rows")
log("    of that ordered result, done with OFFSET 10 (LIMIT -1 in SQLite means")
log("    'no limit', so this returns everything after the 10 most recent).")
log("  - The raw checkouts table is used, so the one duplicated Maadi row")
log("    appears in this output too -- left as-is, to be resolved in Task 2.")
log()
neighborhood = "maadi"
q5 = """
SELECT c.checkout_id, m.member_id,
       m.first_name || ' ' || m.last_name AS member_name,
       c.book_id, c.checkout_date, c.return_date
FROM checkouts c
JOIN members m ON m.member_id = c.member_id
WHERE LOWER(TRIM(m.neighborhood)) = ?
ORDER BY c.checkout_date DESC, c.checkout_id DESC
LIMIT -1 OFFSET 10;
"""
cur.execute(q5, (neighborhood,))
rows5 = cur.fetchall()
for r in rows5:
    log(dict(r))
log(f"Total Maadi checkouts: {len(rows5) + 10}  | shown (beyond most recent 10): {len(rows5)}")

conn.close()

# ---------------------------------------------------------------------------
log()
log("================================================================")
log("REFLECTION: Web Page vs. Database as a Data Source")
log("================================================================")
log("The database and an API both hand data straight to a program in a")
log("fixed, predictable shape -- a SELECT returns known columns and types,")
log("no guessing required. The Reading Kickoff page is built for a person")
log("reading it in a browser, not for a program: it has no schema, no ID")
log("column (a checkout_id wasn't needed for a human just scanning the")
log("table), and no constraints stopping a bad or nonexistent value from")
log("being typed into it. Pulling data from it meant parsing HTML structure")
log("(finding the <table>, checking the header, reading <td> text) instead")
log("of just querying, and treating every value as unverified text until")
log("proven otherwise.")
log()
log("That distinction mattered concretely for this project: the database's")
log("foreign keys guarantee every checkout's member_id is real, but the")
log("Kickoff page has no such guarantee -- and indeed 5 of its 26 signups")
log("reference member_ids that don't exist anywhere in the members table.")
log("A relational database would have caught that at write time; a page")
log("meant for people to read had no mechanism to catch it at all. Knowing")
log("the source was 'read by a person' rather than 'consumed by a program'")
log("is exactly why those rows needed defensive handling instead of a")
log("simple join.")

# Save all answers AND their reasoning to a plain text file
with open(TASK1_ANSWERS_PATH, "w") as f:
    f.write("\n".join(output_lines))

print(f"\nSaved answers and reasoning to {TASK1_ANSWERS_PATH}")

################ Q1: Checkouts per member (incl. zero) ################
Reasoning:
  - Every member must appear in the result, even those with no checkouts,
    so members is the driving table with a LEFT JOIN out to checkouts
    (an INNER JOIN would silently drop members with zero checkouts).
  - COUNT(c.checkout_id) counts only matched checkout rows per member; for
    members with no matches the LEFT JOIN produces NULLs, which COUNT(...)
    correctly reports as 0 rather than NULL.
  - The raw checkouts table is queried directly, with no de-duplication --
    that cleanup is intentionally left for Task 2, so these counts include
    a small number of duplicate-row inflation that Task 2 later corrects.
  - Sorted busiest-first (checkout_count DESC), with member_id as a
    tiebreaker for a stable, reproducible order.

{'member_id': 1034, 'member_name': 'Aya Wahba', 'checkout_count': 25}
{'member_id': 1044, 'member_name': 'Sherif Saleh', 'checkout_count': 21}
{'member_id': 1008, 'mem

### Stage 1 — Members and Checkouts (pure Python, no query tool)

Constraint for this stage: pure Python only -- no SQL JOIN, no
pandas.merge(), no other "query tool" doing the linking for us. We fetch
the two raw tables with plain SELECT * statements and do the matching
ourselves with a dictionary keyed by member_id.

In [4]:
def fetch_all_as_dicts(cursor, table_name):
    """Read an entire table with a bare SELECT * (no JOIN) and return a list
    of plain dicts, one per row."""
    cursor.execute(f"SELECT * FROM {table_name}")
    columns = [d[0] for d in cursor.description]
    return [dict(zip(columns, row)) for row in cursor.fetchall()]


def build_stage1(db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    raw_members = fetch_all_as_dicts(cur, "members")
    raw_checkouts = fetch_all_as_dicts(cur, "checkouts")  # used as-is, duplicates included
    conn.close()

    # Index members by member_id for O(1) lookup -- this dict IS the "join",
    # done by hand instead of by a query engine.
    members_by_id = {m["member_id"]: m for m in raw_members}

    combined = []
    unmatched_checkouts = []  # checkouts whose member_id has no member record
    checkout_counts = {m["member_id"]: 0 for m in raw_members}  # every member starts at 0

    for c in raw_checkouts:
        member = members_by_id.get(c["member_id"])
        if member is None:
            # A checkout referencing a member that doesn't exist would be a
            # mis-link risk; we flag it instead of silently guessing.
            unmatched_checkouts.append(c)
            continue

        row = {
            "checkout_id": c["checkout_id"],
            "source": "database",
            "member_id": member["member_id"],
            "first_name": member["first_name"],
            "last_name": member["last_name"],
            "member_name": f"{member['first_name']} {member['last_name']}",
            "grade": member["grade"],
            "neighborhood": member["neighborhood"],
            "membership_status": member["membership_status"],
            "join_date": member["join_date"],
            "book_id": c["book_id"],
            "checkout_date": c["checkout_date"],
            "return_date": c["return_date"],
        }
        combined.append(row)
        checkout_counts[member["member_id"]] += 1

    # Attach each member's running total onto their own rows, so the total is
    # visible directly on the combined data without a second lookup.
    for row in combined:
        row["member_total_checkouts"] = checkout_counts[row["member_id"]]

    return {
        "combined": combined,
        "checkout_counts": checkout_counts,  # includes members with 0 checkouts
        "members_by_id": members_by_id,
        "raw_checkout_count": len(raw_checkouts),
        "unmatched_checkouts": unmatched_checkouts,
    }


stage1_result = build_stage1()
stage1_combined = stage1_result["combined"]

print(f"Raw checkout rows read: {stage1_result['raw_checkout_count']}")
print(f"Checkout rows in Stage 1 output (no de-duplication applied): {len(stage1_combined)}")
print(f"Checkouts that could not be matched to a member: {len(stage1_result['unmatched_checkouts'])}")
print(f"Members represented (incl. members with 0 checkouts): {len(stage1_result['checkout_counts'])}")
print()
print("Sample combined rows:")
for row in stage1_combined[:3]:
    print(row)
print()
print("Per-member totals (first 5, sorted by member_id):")
for member_id in sorted(stage1_result["checkout_counts"])[:5]:
    print(f"  member_id {member_id}: {stage1_result['checkout_counts'][member_id]} checkouts")

Raw checkout rows read: 391
Checkout rows in Stage 1 output (no de-duplication applied): 391
Checkouts that could not be matched to a member: 0
Members represented (incl. members with 0 checkouts): 80

Sample combined rows:
{'checkout_id': 9263, 'source': 'database', 'member_id': 1047, 'first_name': 'Sara', 'last_name': 'Rashad', 'member_name': 'Sara Rashad', 'grade': None, 'neighborhood': 'Heliopolis', 'membership_status': 'Inactive', 'join_date': '2024-06-25', 'book_id': 517, 'checkout_date': '2024-10-21', 'return_date': '2024-11-07', 'member_total_checkouts': 16}
{'checkout_id': 9340, 'source': 'database', 'member_id': 1072, 'first_name': 'Seif', 'last_name': 'Zaki', 'member_name': 'Seif Zaki', 'grade': 9, 'neighborhood': 'Zamalek', 'membership_status': 'Active', 'join_date': '2025-10-21', 'book_id': 513, 'checkout_date': '2025-08-24', 'return_date': '2025-09-01', 'member_total_checkouts': 14}
{'checkout_id': 9231, 'source': 'database', 'member_id': 1053, 'first_name': 'Adam', 'last

### Stage 2 — Book Details

Book details live in two places that both need to be combined first:
  - database.db -> books table: book_id, title, author
  - books.json          : book_id, genre, pages, publication_year, publisher
Both are keyed by book_id (501-532), so they're merged into one lookup dict
before being attached to each checkout. The checkout row count must stay
exactly the same as Stage 1 -- this stage adds columns, not rows.

In [5]:
def load_books_catalog(db_path=DB_PATH, json_path=BOOKS_JSON_PATH):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    db_books = fetch_all_as_dicts(cur, "books")  # book_id, title, author
    conn.close()

    with open(json_path) as f:
        json_books = json.load(f)  # book_id, genre, pages, publication_year, publisher

    db_books_by_id = {b["book_id"]: b for b in db_books}
    json_books_by_id = {b["book_id"]: b for b in json_books}

    all_ids = set(db_books_by_id) | set(json_books_by_id)
    only_in_db = set(db_books_by_id) - set(json_books_by_id)
    only_in_json = set(json_books_by_id) - set(db_books_by_id)

    books_by_id = {}
    for book_id in all_ids:
        merged = {}
        merged.update(db_books_by_id.get(book_id, {}))
        merged.update(json_books_by_id.get(book_id, {}))
        books_by_id[book_id] = merged

    return {
        "books_by_id": books_by_id,
        "only_in_db": only_in_db,
        "only_in_json": only_in_json,
    }


def build_stage2():
    catalog = load_books_catalog()
    books_by_id = catalog["books_by_id"]

    stage2_rows = []
    unmatched = []

    for row in stage1_combined:
        book = books_by_id.get(row["book_id"])
        new_row = dict(row)  # copy -- don't mutate stage1 data
        if book is None:
            unmatched.append(row["checkout_id"])
            new_row.update({
                "title": None, "author": None, "genre": None,
                "pages": None, "publication_year": None, "publisher": None,
            })
        else:
            new_row.update({
                "title": book.get("title"),
                "author": book.get("author"),
                "genre": book.get("genre"),
                "pages": book.get("pages"),
                "publication_year": book.get("publication_year"),
                "publisher": book.get("publisher"),
            })
        stage2_rows.append(new_row)

    return {
        "combined": stage2_rows,
        "stage1_row_count": len(stage1_combined),
        "stage2_row_count": len(stage2_rows),
        "unmatched_checkout_ids": unmatched,
        "catalog_only_in_db": catalog["only_in_db"],
        "catalog_only_in_json": catalog["only_in_json"],
    }


stage2_result = build_stage2()
stage2_combined = stage2_result["combined"]

print(f"Stage 1 row count: {stage2_result['stage1_row_count']}")
print(f"Stage 2 row count: {stage2_result['stage2_row_count']}")
assert stage2_result["stage1_row_count"] == stage2_result["stage2_row_count"], \
    "Row count changed during Stage 2 -- this should never happen!"
print("Row count unchanged: PASS")
print(f"Checkouts whose book_id had no catalog match: {len(stage2_result['unmatched_checkout_ids'])}")
print(f"Book IDs present only in the DB books table: {sorted(stage2_result['catalog_only_in_db'])}")
print(f"Book IDs present only in books.json: {sorted(stage2_result['catalog_only_in_json'])}")
print()
print("Sample combined rows:")
for row in stage2_combined[:3]:
    print(row)

Stage 1 row count: 391
Stage 2 row count: 391
Row count unchanged: PASS
Checkouts whose book_id had no catalog match: 0
Book IDs present only in the DB books table: []
Book IDs present only in books.json: []

Sample combined rows:
{'checkout_id': 9263, 'source': 'database', 'member_id': 1047, 'first_name': 'Sara', 'last_name': 'Rashad', 'member_name': 'Sara Rashad', 'grade': None, 'neighborhood': 'Heliopolis', 'membership_status': 'Inactive', 'join_date': '2024-06-25', 'book_id': 517, 'checkout_date': '2024-10-21', 'return_date': '2024-11-07', 'member_total_checkouts': 16, 'title': 'Shadows on the Corniche', 'author': 'Hani Nagati', 'genre': 'Mystery', 'pages': 338, 'publication_year': 2015, 'publisher': 'Delta House'}
{'checkout_id': 9340, 'source': 'database', 'member_id': 1072, 'first_name': 'Seif', 'last_name': 'Zaki', 'member_name': 'Seif Zaki', 'grade': 9, 'neighborhood': 'Zamalek', 'membership_status': 'Active', 'join_date': '2025-10-21', 'book_id': 513, 'checkout_date': '2025-0

### Stage 3 — The Reading Kickoff Checkouts

Unlike Stages 1-2, this source isn't a table we can SELECT from -- it's an
HTML page built for a person to read in a browser. We have to parse the table out of the markup ourselves (BeautifulSoup), which is a fundamentally
different, more fragile kind of "read" than a database query or a JSON load
(see the reflection above).

In [6]:
def parse_kickoff_signups(html_path=KICKOFF_HTML_PATH):
    with open(html_path, encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    table = soup.find("table")
    header_cells = [th.get_text(strip=True) for th in table.find("tr").find_all("th")]
    expected_header = ["Member ID", "Book ID", "Checkout Date"]
    if header_cells != expected_header:
        raise ValueError(f"Unexpected table header: {header_cells}")

    signups = []
    data_rows = table.find_all("tr")[1:]  # skip header row
    for tr in data_rows:
        cells = [td.get_text(strip=True) for td in tr.find_all("td")]
        if len(cells) != 3:
            # A row that doesn't match the expected shape -- flag rather than
            # silently drop or silently guess.
            raise ValueError(f"Row with unexpected number of cells: {cells}")
        member_id_str, book_id_str, checkout_date = cells
        signups.append({
            "member_id": int(member_id_str),
            "book_id": int(book_id_str),
            "checkout_date": checkout_date,
        })
    return signups


def build_stage3():
    members_by_id = stage1_result["members_by_id"]
    books_by_id = load_books_catalog()["books_by_id"]

    signups = parse_kickoff_signups()

    # Synthetic checkout_ids that can't collide with the database's real ones
    # (those top out in the 9000s). Prefixed and clearly flagged as such.
    existing_ids = {row["checkout_id"] for row in stage2_combined}
    next_id = 1
    unmatched_members = []
    unmatched_books = []
    kickoff_rows = []

    for signup in signups:
        member = members_by_id.get(signup["member_id"])
        book = books_by_id.get(signup["book_id"])

        if member is None:
            # This is a real finding, not a parsing bug: the Kickoff lets
            # students borrow without a library card, so some signups turn
            # out to reference member_ids that don't exist in `members` at
            # all (1104, 1150, 1201). The task requires every Kickoff row to
            # show up in the result, so we keep the row but leave member
            # fields null rather than inventing a member to link it to --
            # that would be exactly the "wrong member" mistake we're meant
            # to avoid. (This is exactly Task 2, Problem 4.)
            unmatched_members.append(signup)
        if book is None:
            unmatched_books.append(signup)
            continue  # every Kickoff book_id matched in practice; guard anyway

        synthetic_id = f"RK-{next_id:03d}"
        while synthetic_id in existing_ids:
            next_id += 1
            synthetic_id = f"RK-{next_id:03d}"
        next_id += 1
        existing_ids.add(synthetic_id)

        row = {
            "checkout_id": synthetic_id,
            "source": "reading_kickoff",
            "member_id": signup["member_id"],
            "first_name": member["first_name"] if member else None,
            "last_name": member["last_name"] if member else None,
            "member_name": f"{member['first_name']} {member['last_name']}" if member else None,
            "grade": member["grade"] if member else None,
            "neighborhood": member["neighborhood"] if member else None,
            "membership_status": member["membership_status"] if member else "Not a registered member",
            "join_date": member["join_date"] if member else None,
            "book_id": signup["book_id"],
            "checkout_date": signup["checkout_date"],
            "return_date": None,  # not recorded for Kickoff loans -- due at summer's end
            "member_total_checkouts": None,  # recomputed below across the full dataset
            "title": book.get("title"),
            "author": book.get("author"),
            "genre": book.get("genre"),
            "pages": book.get("pages"),
            "publication_year": book.get("publication_year"),
            "publisher": book.get("publisher"),
        }
        kickoff_rows.append(row)

    final_rows = stage2_combined + kickoff_rows

    # member_total_checkouts must reflect the FULL combined dataset now,
    # not just the database portion computed back in Stage 1.
    totals = {}
    for row in final_rows:
        totals[row["member_id"]] = totals.get(row["member_id"], 0) + 1
    for member_id in members_by_id:
        totals.setdefault(member_id, 0)
    for row in final_rows:
        row["member_total_checkouts"] = totals[row["member_id"]]

    return {
        "combined": final_rows,
        "stage2_row_count": len(stage2_combined),
        "kickoff_signup_count": len(signups),
        "kickoff_rows_added": len(kickoff_rows),
        "unmatched_members": unmatched_members,
        "unmatched_books": unmatched_books,
        "final_row_count": len(final_rows),
        "member_totals": totals,
    }


stage3_result = build_stage3()
final_combined = stage3_result["combined"]

print(f"Reading Kickoff signups parsed from the web page: {stage3_result['kickoff_signup_count']}")
print(f"Kickoff signups successfully linked and added: {stage3_result['kickoff_rows_added']}")
print(f"Kickoff signups with an unmatched member_id: {len(stage3_result['unmatched_members'])}")
print(f"Kickoff signups with an unmatched book_id: {len(stage3_result['unmatched_books'])}")
assert stage3_result["kickoff_signup_count"] == stage3_result["kickoff_rows_added"], \
    "Not every Reading Kickoff signup made it into the combined dataset!"
print("Every Kickoff signup accounted for: PASS")
print()
print(f"Stage 2 (database-only) row count: {stage3_result['stage2_row_count']}")
print(f"Final combined row count (no cleaning applied): {stage3_result['final_row_count']}")
print()
print("Sample Reading Kickoff rows in the final combined dataset:")
for row in final_combined:
    if row["source"] == "reading_kickoff":
        print(row)
        break

# Write out the (still uncleaned) combined dataset -- this is the file Task 2
# picks up and works on.
with open(TASK1_JSON_PATH, "w") as f:
    json.dump(final_combined, f, indent=2, default=str)

fieldnames = list(final_combined[0].keys())
with open(TASK1_CSV_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(final_combined)

db_rows = sum(1 for r in final_combined if r["source"] == "database")
kickoff_rows = sum(1 for r in final_combined if r["source"] == "reading_kickoff")
kickoff_unmatched = sum(
    1 for r in final_combined if r["source"] == "reading_kickoff" and r["member_name"] is None
)

print(f"\nCombined dataset written to {TASK1_JSON_PATH} and {TASK1_CSV_PATH}")
print(f"Total rows: {len(final_combined)}")
print(f"  from the database: {db_rows}")
print(f"  from the Reading Kickoff page: {kickoff_rows}")
print(f"    of which unmatched to a registered member: {kickoff_unmatched}")

Reading Kickoff signups parsed from the web page: 26
Kickoff signups successfully linked and added: 26
Kickoff signups with an unmatched member_id: 5
Kickoff signups with an unmatched book_id: 0
Every Kickoff signup accounted for: PASS

Stage 2 (database-only) row count: 391
Final combined row count (no cleaning applied): 417

Sample Reading Kickoff rows in the final combined dataset:
{'checkout_id': 'RK-001', 'source': 'reading_kickoff', 'member_id': 1026, 'first_name': 'Nada', 'last_name': 'Saleh', 'member_name': 'Nada Saleh', 'grade': 7, 'neighborhood': 'Nasr City', 'membership_status': 'inactive', 'join_date': '2023-11-16', 'book_id': 522, 'checkout_date': '2025-07-11', 'return_date': None, 'member_total_checkouts': 4, 'title': 'The Puzzle Merchant', 'author': 'Karim Elwy', 'genre': 'Mystery', 'pages': 104, 'publication_year': 2016, 'publisher': 'Cairo Young Readers'}

Combined dataset written to task1_combined_data.json and task1_combined_data.csv
Total rows: 417
  from the databa